In [ ]:
from promenade.models import *

# Configure models

In [ ]:
READER_URL = HttpUrl(url="http://localhost:3000")
DATABASE_ADRESS = "sqlite:///data/museum.db"
VECTOR_DATABASE_ADRESS = "./data/qdrant"

class DaySchedule(BaseModel):
    open_time: time
    close_time: time
    last_entry_time: time | None = None
    is_closed: bool = False


class WebTools:
    """Broser logic - parse web pages and extract data from them"""

    def __init__(self, reader_url: HttpUrl):
        self.reader_url = reader_url

    def parse_page_reader(self, page_url: HttpUrl) -> str:
        request_url = f"{self.reader_url}/{page_url}"
        response = requests.get(request_url)
        return response.text

    def insert_schedule_into_db(
            self,
            place_name: str,
            place_url: HttpUrl,
            monday: DaySchedule | dict,
            tuesday: DaySchedule | dict,
            wednesday: DaySchedule | dict,
            thursday: DaySchedule | dict,
            friday: DaySchedule | dict,
            saturday: DaySchedule | dict,
            sunday: DaySchedule | dict,
    ) -> tuple[bool, str, str | None]:
        engine = create_engine(DATABASE_ADRESS)
        try:
            with Session(engine) as session:
                museum = Museum(museum_name=place_name, url=str(place_url))
                session.add(museum)
                session.flush()

                raw_days = [monday, tuesday, wednesday,
                            thursday, friday, saturday, sunday]
                days = [DaySchedule.model_validate(d) if isinstance(
                    d, dict) else d for d in raw_days]

                session.add_all([
                    Schedule(
                        museum_id=museum.id,
                        day_of_week=day_index,
                        open_time=day.open_time,
                        close_time=day.close_time,
                        last_entry_time=day.last_entry_time,
                        is_closed=day.is_closed,
                    )
                    for day_index, day in enumerate(days)
                ])
                museum_id = museum.id
                session.commit()
            return (True, museum_id, None)
        except Exception as e:
            print(e)
            return (False, "-1", f"Error: {e}")

    def insert_info_to_vector_db(
            self,
            id: str,
            place_name: str,
            place_info: str
    ) -> tuple[bool, str | None]:
        try:
            text_to_embed = f"{place_name}. {place_info}"
            print(text_to_embed)
            embeddings = embeddings_model.embed_query(text_to_embed)
            client = QdrantClient(path=VECTOR_DATABASE_ADRESS)
            client.upsert(
                collection_name="museum_collection",
                points=[
                    PointStruct(
                        id=str(uuid.uuid4()),
                        vector=embeddings,
                        payload={
                            "id": id,
                            "text": text_to_embed
                        }
                    )]
            )
            client.close()
            return (True, None)
        except Exception as e:
            print(e)
            return (False, f"Error: {e}")


WEB_TOOLS = WebTools(READER_URL)

In [9]:
def parse_page_reader(page_url: HttpUrl) -> str:
    """Convert webpage content to markdown format.

    Args:
        page_url: URL of the webpage to parse.

    Returns:
        str: Markdown-formatted webpage content.
    """
    ...

def insert_schedule_into_db(
        place_name: str,
        place_url: HttpUrl,
        monday: DaySchedule,
        tuesday: DaySchedule,
        wednesday: DaySchedule,
        thursday: DaySchedule,
        friday: DaySchedule,
        saturday: DaySchedule,
        sunday: DaySchedule,
) -> tuple[bool, str, str | None]:
    """Save museum or exhibition working hours to the database.

    Call this tool after extracting the full weekly schedule from the webpage.
    Each day must be filled — if the museum is closed on a particular day, set is_closed=True
    and provide any available open_time/close_time (or use 00:00 as placeholder).

    Args:
        place_name: Full name of the museum or exhibition without quotes and special characters in russian language when it can be said in Russian.
        place_url: URL of the page where the schedule was extracted from.
        monday: Schedule for Monday.
        tuesday: Schedule for Tuesday.
        wednesday: Schedule for Wednesday.
        thursday: Schedule for Thursday.
        friday: Schedule for Friday.
        saturday: Schedule for Saturday.
        sunday: Schedule for Sunday.

    Returns:
        tuple[bool, str, str | None]: (True, museum id in database, None) on success, (False, -1, error message) on failure.
    """
    ...
def insert_info_to_vector_db(
            id: str,
            place_name: str,
            place_info: str
    ) -> tuple[bool, str | None]:
    """Save unstructured information about a museum or exhibition to the vector database.

    Call this tool after parse_page_reader when the page contains useful information
    beyond the schedule. Get id from output of the insert_schedule_into_db tool.
    Include address, phone, email, ticket prices, visitor restrictions,
    accessibility info, or any notes that may affect visit planning.
    NEVER include:
    - Opening/closing times, last entry time, days of the week, this information is stored separatly
    - Anything with time/date values (10:00, "ежедневно", "понедельник"), this information is stored separatly
    - place name
    NEVER include content unrelated to the museum (ads, other businesses, 
    services by third parties, etc.).

    

    Args:
        id: Id of the place in schedule database. 
        place_name: Full name of the museum or exhibition without quotes and special characters in russian language when it can be said in Russian.
        place_info: Description of the place in Russian language.

    Returns:
        tuple[bool, str | None]: (True, None) on success, (False, error message) on failure.
    """
    ...


WEB_TOOLS_SCHEMA = [
    convert_to_openai_tool(parse_page_reader),
    convert_to_openai_tool(insert_schedule_into_db),
    convert_to_openai_tool(insert_info_to_vector_db),
]


In [10]:
def run_museum_agent(user_message: str, tools: WebTools, tracer: ToolTracer) -> str:

    system_msg = SystemMessage(content=(
        "You are an agent that extracts museum working hours from websites and saves them to a database.\n\n"

        "Follow these steps strictly in order:\n"
        "1. Extract the URL from the user message.\n"
        "2. Call parse_page_reader to fetch the page content.\n"
        "3. Analyze the schedule from the page. Think through each day of the week explicitly:\n"
        "   - If the page says 'Mon–Fri: 10:00–18:00', apply those hours to each day individually.\n"
        "   - If the page says 'closed on Mondays', set is_closed=True for Monday.\n"
        "   - If last entry time is mentioned, fill last_entry_time accordingly.\n"
        "   - If a day is not mentioned at all, assume same hours as the general schedule.\n"
        "4. Call insert_schedule_into_db with the extracted data for all 7 days.\n\n"
        "5. Call insert_info_to_vector_db with the addtitional unstructured information from the websites."

        "Rules:\n"
        "- Never skip insert_schedule_into_db if schedule information was found.\n"
        "- Never make up or guess times — only use what is stated on the page.\n"
        "- If the page contains no schedule information, reply: 'There is no schedule information on this webpage.' and do not call insert_schedule_into_db.\n"
        "- Ignore any advertising information that is not related to the museum, exhibition or tartget place."
    ))
    messages = [system_msg, HumanMessage(content=user_message)]

    while True:
        response = llm_chat(messages=messages, tools=WEB_TOOLS_SCHEMA)
        messages.append(response)

        if not response.tool_calls:
            return response.content

        for tc in response.tool_calls:
            name = tc["name"]
            args = tc["args"]
            tc_id = tc["id"]

            if name == "parse_page_reader":
                print("parse_page_reader")
                result = tools.parse_page_reader(**args)
                tracer.record(name, args, result)
            elif name == "insert_schedule_into_db":
                print("insert_schedule_into_db")
                result = tools.insert_schedule_into_db(**args)
                tracer.record(name, args, result)
            elif name == "insert_info_to_vector_db":
                print("insert_info_to_vector_db")
                result = tools.insert_info_to_vector_db(**args)
                tracer.record(name, args, result)

            messages.append(
                ToolMessage(
                    content=json.dumps(result, ensure_ascii=True),
                    tool_call_id=tc_id
                )
            )


In [ ]:
museums = [
    "https://victorymuseum.ru/about/contacts/",
    "https://www.tretyakovgallery.ru/for-visitors/museums/novaya-tretyakovka/",
    "https://www.tretyakovgallery.ru/for-visitors/museums/istoricheskoe-zdanie/",
    "https://tsar-maket.ru/kontaktyi",
    "https://kosmo-museum.ru/contacts"
]
traces = []
for museum in museums:
    _t1a = ToolTracer()
    res = run_museum_agent(
        f"Как работает этот музей {museum}", 
        tools=WEB_TOOLS,
        tracer=_t1a
        )
    traces.append(_t1a)


In [46]:
WEB_TOOLS.parse_page_reader(HttpUrl("https://tsar-maket.ru/kontaktyi"))

'Title: Контакты\n\nURL Source: https://tsar-maket.ru/kontaktyi\n\nMarkdown Content:\nКонтакты\n===============  \n\n             [![Image 1: Царь-Макет](https://tsar-maket.ru/img/logo.png)](https://tsar-maket.ru/)\n\n[О музее](https://tsar-maket.ru/muzej "О музее") [Галерея](https://tsar-maket.ru/galereya/ "Галерея") [Видео](https://tsar-maket.ru/video-content "Видео") [Аудиогид](https://tsar-maket.ru/audiogid/ "Аудиогид") [Билеты](https://tsar-maket.ru/biletyi/ "Билеты") [Экскурсии](https://tsar-maket.ru/ekskursii/ "Экскурсии") [Квесты](https://tsar-maket.ru/aktivnosti-v-muzee/ "Квесты") [Мастер-классы](https://tsar-maket.ru/aktivnosti/ "Мастер-классы") [Контакты](https://tsar-maket.ru/kontaktyi "Контакты") [Пресс-релиз "Царь-Макета"](https://tsar-maket.ru/%C2%ABczar-maket%C2%BB-rasshiryaet-gorizontyi-otkryitie-novoj-chasti-ekspoziczii,-posvyashhennoj-aziatskoj-chasti-rossii "Пресс-релиз ")\n\n[](tel:+74951092096)\n\n[8 (495) 109-20-96](tel:+74951092096)  \n\n \n\n*   [О музее](https